### Simulation Investigation

Two variations of simulations; first utilizes KDE as a prior while latter directly simualtes cp

In [123]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


path='/home/jbohn/jupyter/personal/'
sys.path.append(f'{path}TAQ-Query-Scripts/data')
sys.path.append(f'{path}Changepoint_LOB/Lasso/CPD')
from kernel_smoother import smooth_series, cv_block_size, mom_kde
from adaptive_tf_ir import l1tf_adaptive_ir, Dmat

In [166]:
trades=pd.read_csv(f'{path}TAQ-Query-Scripts/data/cleaned_trades.csv',index_col=0).dropna()

trades=trades[trades.index < "2020-01-02 09:31:30"]
trades=trades[trades.index> "2020-01-02 09:30:30"]
trades['Trade_Price']=np.log(trades['Trade_Price'])
trades.index=pd.to_datetime(trades.index)

### Step 1; Generate Prior

Use volume data; normalizing index to difference in seconds

In [ ]:
original_index=trades['Trade_Volume'].index

volume_index=[(i -original_index[0]).total_seconds() for i in original_index]
volume_prior=trades['Trade_Volume'].values

In [ ]:
fig,ax=plt.subplots(figsize=(12,10))

plt.title("Kernel Smoothed Series")
plt.plot(volume_index,volume_prior,label="Original Prior")
plt.legend()

### Step 2; Given Prior Apply Kernel Smoothing

- Apply Kernel smoothing across our penalty specifically
    - Gaussian Kernels using Scott's/ Silverman for bandwidth estimation
    - Robustness corrections (MOM)


In [ ]:
kde_smooth,optimal_bw=smooth_series(volume_prior,volume_index)

In [ ]:
block_params=np.arange(1,100,10)
optimal_mom_kde_bw,optimal_mse=cv_block_size(volume_prior,kde_smooth,volume_prior,optimal_bw,block_params,True)

In [ ]:
optimal_mom_kde=mom_kde(kde_smooth,volume_index,optimal_mom_kde_bw,optimal_bw)

def min_max_norm(series):
    return (series-np.min(series))/(np.max(series)-np.min(series))

optimal_mom_kde_norm=min_max_norm(optimal_mom_kde)

In [ ]:
fig,ax=plt.subplots(figsize=(12,10))

plt.title("Kernel Smoothed Series")
plt.plot(volume_index,kde_smooth,label="Original KDE")
plt.plot(volume_index,optimal_mom_kde,label="Optimal MOM KDE")
plt.legend()

### Step 3; Sample Exponential distribution scaled by prior for sparsity enforcing param

In [ ]:
def sample_from_prior(smooth_prior):
    samples=[]
    for x_i in smooth_prior:
        scale=2/(x_i**2)
        s_i=(x_i/2)*np.random.exponential(scale=scale)
        
        samples.append(s_i)
    
    return samples

In [ ]:
sparse_samples=sample_from_prior(optimal_mom_kde)

In [ ]:
plt.title("Sample Laplace Distribution of Prior")
bins=plt.hist(sparse_samples,bins=20)

### Step 4 ; Sample $X_t$ given previous $X_i$ and laplace scale on variance

- Issue in sampling with numerical overflow
- Specifically small variance gives our large oscillations in x values


In [ ]:
def sample_price_series(sparse_params,variance=1,init=0):
    
    obs=[]
    
    for count,param in enumerate(sparse_params):
        mu=sparse_params[count-1]-sparse_params[count-2]
        scale=np.sqrt(param)*variance
        
        sample=np.random.normal(loc=mu,scale=scale,size=1)[0]
        
        obs.append(sample) 
    
    returns=np.cumprod(np.array(obs)+1)+init
    return returns

In [ ]:
price_series=sample_price_series(sparse_samples,variance=0.1,init=100)


In [ ]:
fig,ax=plt.subplots(figsize=(12,10))
plt.plot(price_series)
plt.title("Sampled Noisy Series")
plt.xlabel('Time')
plt.ylabel("Price")

Take cumulative product; scale by constant  

In [ ]:
trend_filter_series,status,D=l1tf_adaptive_ir(price_series.reshape(-1,1),lambda_p=100)

In [ ]:
fig,ax=plt.subplots(figsize=(12,10))
plt.plot(price_series,label='Noisy')
plt.plot(trend_filter_series,label='Smooth',lw=5)
plt.title("Original and Filtered")
plt.xlabel("Time")
plt.ylabel('Price')
plt.legend()

### Extract Changepoints

Changepoints in the underlying trend can be extracted via matrix multiplication of second difference matrix

In [ ]:
def extract_cp(smooth,difference_order=2,threshold=1e-4):
    diff_mat=Dmat(len(smooth),2).todense()
    diff=np.dot(diff_mat,smooth).reshape(1,-1)[0]
    
    x,y,index=np.where([abs(diff)>threshold])
    return index

In [ ]:
changepoints=extract_cp(trend_filter_series)

In [ ]:
fig,ax=plt.subplots(figsize=(12,10))
plt.plot(price_series,label='Noisy')
plt.plot(trend_filter_series,label='Smooth',lw=5)
plt.scatter(changepoints,trend_filter_series[changepoints],color='black',s=250)
plt.title("Original & Filtered with Changepoints")
plt.xlabel("Time")
plt.ylabel('Price')
plt.legend()